# MotionJSON Local UI with hosted SAM provider setup

This notebook launches the MotionJSON Local UI inside a Google Colab runtime and opens `/ui/` through Colab's built-in notebook port proxy. It installs the optional hosted SAM vendor dependencies so the UI can connect to Roboflow SAM3, Replicate SAM2 video, Fal SAM3 image, or custom SAM2/SAM3-compatible endpoints after explicit cost and privacy opt-in.

Provider keys are read from Colab userdata when available, with an interactive `getpass` fallback. You can also leave these blank and paste temporary credentials into the UI provider settings form. Do not save private videos, provider credentials, or shared notebook outputs containing secrets.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

repo_url = "https://github.com/ptse8204/json-animated-video.git"
workdir = Path("/content/json-animated-video")

if not workdir.exists():
    subprocess.run(["git", "clone", repo_url, str(workdir)], check=True)
else:
    subprocess.run(["git", "-C", str(workdir), "fetch", "--depth", "1", "origin", "main"], check=True)
    subprocess.run(["git", "-C", str(workdir), "checkout", "main"], check=True)
    subprocess.run(["git", "-C", str(workdir), "pull", "--ff-only"], check=True)

os.chdir(workdir)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-e", ".[ui,hosted-segmentation,hosted-sam3,hosted-sam-vendors]"],
    check=True,
)


In [ ]:
from getpass import getpass

def colab_user_secret(name: str) -> str:
    try:
        from google.colab import userdata
        value = userdata.get(name)
    except Exception:
        value = None
    return (value or "").strip()

for env_name in ["ROBOFLOW_API_KEY", "REPLICATE_API_TOKEN", "FAL_KEY"]:
    value = colab_user_secret(env_name)
    if not value:
        value = getpass(f"{env_name} (leave blank to skip): ").strip()
    if value:
        os.environ[env_name] = value

configured_names = [name for name in ["ROBOFLOW_API_KEY", "REPLICATE_API_TOKEN", "FAL_KEY"] if os.environ.get(name)]
print("Configured provider secret names:", configured_names or "none")


In [ ]:
subprocess.run([sys.executable, "examples/make_demo_video.py", "--out", "examples/demo_red_ball.mp4"], check=True)
subprocess.run([sys.executable, "-m", "motionjson.cli", "backend", "diagnostics", "--text"], check=True)
print("Demo video path to register in the UI: examples/demo_red_ball.mp4")


In [ ]:
import time
from google.colab import output

port = 8766
ui_proc = subprocess.Popen(
    ["motionjson", "ui", "--no-open", "--mock", "--host", "127.0.0.1", "--port", str(port)],
    cwd=str(workdir),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)
time.sleep(5)
print("MotionJSON UI started with: motionjson ui --no-open --mock --host 127.0.0.1 --port", port)
print("Use Provider settings in the UI to choose Replicate SAM2 video, Roboflow SAM3, Fal SAM3 image, or a custom endpoint.")
output.serve_kernel_port_as_iframe(port, path="/ui/", height=900)


In [ ]:
from google.colab import output

output.serve_kernel_port_as_window(8766, path="/ui/")


In [ ]:
if ui_proc.poll() is None and ui_proc.stdout is not None:
    print("UI server is still running. Recent logs will appear here only after the process writes more output.")
else:
    print("UI server exited with code", ui_proc.returncode)


In [ ]:
if 'ui_proc' in globals() and ui_proc.poll() is None:
    ui_proc.terminate()
    print("MotionJSON UI stopped.")
